# SSHR trên Kaggle — Python 3.10, AMP và resume

1. Upload notebook này lên Kaggle. Settings: **GPU**, **Internet On**. Một GPU được sử dụng; không cộng VRAM hai T4.
2. Dataset và weights tự tải từ [nttduc/wsss-pathology](https://huggingface.co/datasets/nttduc/wsss-pathology/tree/main). Chọn `DATASET='luad'` hoặc `'bcss'`; không cần upload data/weights lên Kaggle.
3. Thêm Kaggle Secret **HF_TOKEN** (token Hugging Face có quyền đọc), bật quyền truy cập cho notebook. Cell đăng nhập hỗ trợ Secrets hoặc nhập token ẩn (`prompt`); chọn `skip` nếu muốn tải public không đăng nhập. Sửa cấu hình nếu cần rồi Run All. Code repo đã nhúng trong notebook, bao gồm hỗ trợ resume; không cần GitHub cập nhật.
4. Mặc định **train lại từ đầu**, chạy liên tục với `SESSION_EPOCHS=0`, `SESSION_HOURS=25`, tổng `MAX_EPOCHS=25`. Không tự nạp checkpoint cũ. Output mỗi run có tên riêng. Theo log đo được ~1,24 giờ/epoch, 25 giờ tương đương khoảng 20 epoch; 25 epoch có thể cần khoảng 31 giờ. Giới hạn mềm chỉ được kiểm tra cuối epoch nên có thể vượt ngân sách gần một epoch.
5. **Kaggle công bố tối đa 12 giờ cho mỗi phiên GPU**, dù notebook đặt 25 giờ ([tài liệu](https://www.kaggle.com/docs/notebooks)). Cấu hình này không kéo dài phiên Kaggle. Checkpoint vẫn lưu cuối mỗi epoch. Dùng **Save Version → Save & Run All** để tạo output bền vững. Phiên tiếp: Add Input output của phiên trước, đặt `RESUME` tới `training_resume.pth` và Run All. `/kaggle/working` của phiên interactive không phải bản sao lưu bền vững.

Checkpoint chỉ lưu sau epoch. Budget giờ là giới hạn mềm ở cuối epoch, không bảo đảm tránh timeout nếu epoch quá dài. Không dùng cách giữ kết nối để vượt quota. Không thay batch/seed/AMP/lịch LR giữa các phiên resume. Chỉ nạp checkpoint do bạn tạo hoặc tin cậy.

Môi trường MXNet riêng dùng Python 3.10 + NumPy 1.23.5 để chuyển `.params` sang NumPy rồi PyTorch. Môi trường train dùng Python 3.10, torch 2.9.0 và requirements của repo, NumPy 2.2.6; không import MXNet khi train. Như vậy tránh xung đột NumPy giữa MXNet và scikit-image/OpenCV hiện tại. CUDA wheel cu126 cần driver tương thích; cell kiểm tra sẽ báo lỗi trước khi train.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, time

DATASET = 'luad'                       # 'luad' hoặc 'bcss'
HF_REPO_ID = 'nttduc/wsss-pathology'
HF_REVISION = 'db41ca8ad5fed44d81ce8e6acb3276647a6d9b47'  # giữ nguyên khi resume
DATA_SOURCE = ''                       # để trống: tự tải ZIP theo DATASET; hoặc đường dẫn local
TRAIN_SUBDIR = 'training'              # đổi thành 'train' hoặc 'train/img' theo dữ liệu
TEST_SUBDIR = 'test'
WEIGHTS = ''                          # để trống: tự tải initialization weights từ cùng HF repo
RESUME = ''                           # trống = train từ đầu; chỉ resume khi điền rõ đường dẫn
SEED = 11
MAX_EPOCHS = 25                        # tổng toàn bộ run; giữ nguyên khi resume
SESSION_EPOCHS = 0                     # chạy liên tục, không giới hạn epoch mỗi phiên
SESSION_HOURS = 25.0                   # ngân sách mềm cuối epoch; KHÔNG vượt giới hạn phiên Kaggle
BATCH_SIZE = 4                        # khởi đầu bảo thủ; nếu đủ VRAM có thể thử 20 như tác giả
NUM_WORKERS = 4
AMP = 'auto'                          # auto -> bf16 nếu hỗ trợ, ngược lại fp16
RUN_EVAL = False                       # chỉ evaluate khi đủ 25 epoch; tốn thêm thời gian

WORK = Path('/kaggle/working')
REPO = WORK / 'SSHR'
# Mỗi lần chạy cell cấu hình tạo output riêng, giữ checkpoint cũ nguyên vẹn.
RUN_ID = str(time.time_ns())
OUT = WORK / f'checkpoints_{DATASET}_seed{SEED}_{RUN_ID}'
assert DATASET in ('luad', 'bcss')
assert SESSION_EPOCHS >= 0 and MAX_EPOCHS > 0 and BATCH_SIZE > 0
if RESUME:
    assert Path(RESUME).is_file(), RESUME

def run(args, **kwargs):
    print(' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True, **kwargs)


In [ ]:
# Snapshot code local: không cần clone repo hay upload code riêng.
import base64, io, zipfile
REPO.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as archive:
    archive.extractall(REPO)
print('Đã chuẩn bị code:', REPO)


In [ ]:
# Python 3.10 riêng; không thay kernel Python của Kaggle.
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
assert UV, 'Không tìm thấy uv'
run([UV, 'python', 'install', '3.10'])
ENV = Path('/tmp/sshr-train-env')
run([UV, 'venv', '--python', '3.10', '--allow-existing', ENV])
PYTHON = ENV / 'bin/python'
os.environ['MPLBACKEND'] = 'Agg'

# Driver do Kaggle quản lý; Python riêng vẫn dùng cùng GPU/driver.
_smi = shutil.which('nvidia-smi')
if _smi:
    _result = subprocess.run([_smi], capture_output=True, text=True)
    print(_result.stdout, flush=True)
    if _result.stderr:
        print(_result.stderr, flush=True)
else:
    print('Không tìm thấy nvidia-smi. Kiểm tra Settings > Accelerator > GPU.', flush=True)

run([UV, 'pip', 'install', '--python', PYTHON,
     'torch==2.9.0', 'torchvision==0.24.0', 'torchaudio==2.9.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])

# Bản headless giữ API cv2, tránh phụ thuộc GUI/libGL trên máy chủ.
# Gỡ bản GUI của lần chạy cũ trước khi cài, tránh hai package cùng ghi cv2.
_installed = subprocess.check_output(
    [str(UV), 'pip', 'list', '--python', str(PYTHON), '--format', 'json'], text=True)
_gui_packages = [x['name'] for x in json.loads(_installed)
                 if x['name'] in ('opencv-python', 'opencv-contrib-python', 'opencv-contrib-python-headless')]
if _gui_packages:
    run([UV, 'pip', 'uninstall', '--python', PYTHON, *_gui_packages])
_requirements = (REPO / 'requirements.txt').read_text().splitlines()
_requirements = [line.replace('opencv-python==', 'opencv-python-headless==')
                 for line in _requirements]
_kaggle_requirements = REPO / '_requirements_kaggle.txt'
_kaggle_requirements.write_text('\n'.join(_requirements) + '\n')
run([UV, 'pip', 'install', '--python', PYTHON, '-r', _kaggle_requirements, 'numpy==2.2.6'])
# Reinstall after switching cv2 providers: shared files may have been removed.
if _gui_packages:
    _opencv_req = next(x for x in _requirements if x.startswith('opencv-python-headless=='))
    run([UV, 'pip', 'install', '--python', PYTHON, '--reinstall-package', 'opencv-python-headless', _opencv_req])
run([UV, 'pip', 'check', '--python', PYTHON])

def check_environment():
    failures = []
    for module in ('torch', 'torchvision', 'cv2', 'skimage', 'matplotlib'):
        result = subprocess.run(
            [str(PYTHON), '-u', '-c',
             f"import {module}; print('{module}:', getattr({module}, '__version__', 'OK'))"],
            capture_output=True, text=True)
        print(result.stdout, end='', flush=True)
        if result.returncode:
            print(f'IMPORT FAILED: {module} (exit {result.returncode})', flush=True)
            print(result.stderr or '(không có stderr)', flush=True)
            failures.append(module)
        elif result.stderr:
            print(result.stderr, flush=True)
    if failures:
        raise RuntimeError('Import thất bại: ' + ', '.join(failures)
                           + '. Xem lỗi gốc IMPORT FAILED ở trên; đây chưa phải kết luận về GPU.')
    cuda_probe = """
import torch
print('PyTorch:', torch.__version__, 'CUDA build:', torch.version.cuda, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
print('Compiled GPU architectures:', torch.cuda.get_arch_list(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: kiểm tra Kaggle GPU accelerator và driver ở nvidia-smi.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('Compute capability:', torch.cuda.get_device_capability(0), flush=True)
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30, flush=True)
x = torch.ones((32, 32), device='cuda')
y = x @ x
assert y[0, 0].item() == 32
print('CUDA allocation + matmul: OK', flush=True)
"""
    result = subprocess.run([str(PYTHON), '-u', '-c', cuda_probe], capture_output=True, text=True)
    print(result.stdout, end='', flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    if result.returncode:
        raise RuntimeError('CUDA kiểm tra thất bại. Xem lỗi gốc ở trên; '
                           'nếu driver/kiến trúc GPU không tương thích, cần chọn GPU hoặc wheel phù hợp.')

check_environment()
AMP_USED = subprocess.check_output([str(PYTHON), '-c',
    "import torch; print('bf16' if torch.cuda.is_bf16_supported() else 'fp16')"], text=True).strip() if AMP == 'auto' else AMP
print('AMP:', AMP_USED)


In [ ]:
# Đăng nhập Hugging Face. Không ghi token trực tiếp vào notebook.
# Kaggle: Add-ons > Secrets > tạo HF_TOKEN và bật quyền truy cập cho notebook.
# Save & Run All: dùng kaggle_secret vì không có hộp nhập tương tác.
HF_AUTH_METHOD = 'kaggle_secret'        # 'kaggle_secret', 'prompt', hoặc 'skip'
HF_SECRET_NAME = 'HF_TOKEN'

import getpass
os.environ['HF_HOME'] = '/tmp/sshr-hf-auth'  # token/cache nằm ngoài notebook output
run([UV, 'pip', 'install', '--python', PYTHON, 'huggingface_hub>=0.34,<2'])
_hf_token = None
if HF_AUTH_METHOD == 'kaggle_secret':
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(
            'Thêm Kaggle Secret HF_TOKEN và bật quyền truy cập, '
            'hoặc đổi HF_AUTH_METHOD thành prompt (interactive) / skip (public).'
        ) from None
elif HF_AUTH_METHOD == 'prompt':
    _hf_token = getpass.getpass('Hugging Face token (nhập ẩn): ')
elif HF_AUTH_METHOD != 'skip':
    raise ValueError('HF_AUTH_METHOD phải là kaggle_secret, prompt hoặc skip')

if HF_AUTH_METHOD != 'skip':
    if not _hf_token or not _hf_token.strip():
        raise ValueError('Token trống. Nhập token hoặc chọn skip.')
    # Truyền qua stdin: không đặt token trong command, output, config hay source.
    _auth_code = """
import sys
from huggingface_hub import login, HfApi
login(token=sys.stdin.read().strip(), add_to_git_credential=False)
print('Hugging Face account:', HfApi().whoami()['name'])
"""
    try:
        subprocess.run([str(PYTHON), '-c', _auth_code], input=_hf_token.strip(),
                       text=True, check=True)
    finally:
        _hf_token = None
else:
    print('Bỏ qua đăng nhập mới; nguồn public vẫn tải được.')


In [ ]:
# Tải ZIP gốc để giữ filename chứa image-level labels và cặp img/mask.
# load_dataset(...) tạo Dataset abstraction; trainer này cần cây thư mục gốc.
# Chỉ tải dataset đã chọn + initialization weights (không tải weights khi resume).
HF_RESOLVED_REVISION = None
if not DATA_SOURCE or (not RESUME and not WEIGHTS):
    fetch_script = REPO / '_fetch_hf_data.py'
    fetch_script.write_text("""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
repo, revision, dataset, need_data, need_weights, result_path = sys.argv[1:]
info = HfApi().dataset_info(repo, revision=revision)
sha = info.sha
available = {f.rfilename for f in info.siblings}
files = {}
if need_data == '1':
    files['data'] = {'luad': 'LUAD-HistoSeg.zip', 'bcss': 'BCSS-WSSS.zip'}[dataset]
if need_weights == '1':
    files['weights'] = 'ilsvrc-cls_rna-a1_cls1000_ep-0001.params'
result = {'repo_id': repo, 'revision': sha}
for key, name in files.items():
    if name not in available:
        raise FileNotFoundError(f'{name} missing from {repo}@{sha}')
    result[key] = hf_hub_download(repo_id=repo, repo_type='dataset', filename=name,
        revision=sha, local_dir='/tmp/sshr-hf/' + sha)
Path(result_path).write_text(json.dumps(result))
""")
    manifest = Path('/tmp/sshr-hf-download.json')
    run([PYTHON, fetch_script, HF_REPO_ID, HF_REVISION, DATASET,
         int(not DATA_SOURCE), int(not RESUME and not WEIGHTS), manifest])
    downloaded = json.loads(manifest.read_text())
    DATA_SOURCE = DATA_SOURCE or downloaded.get('data', '')
    WEIGHTS = WEIGHTS or downloaded.get('weights', '')
    HF_RESOLVED_REVISION = downloaded['revision']
    print('Hugging Face revision:', HF_RESOLVED_REVISION)
assert Path(DATA_SOURCE).exists(), f'Dataset không tồn tại: {DATA_SOURCE}'
assert RESUME or (WEIGHTS and Path(WEIGHTS).is_file()), 'Không tìm thấy initialization weights'
print('Dataset:', DATA_SOURCE)
if not RESUME:
    print('Weights:', WEIGHTS)


In [ ]:
# Giải nén chỉ khi cần. DATA_ROOT cuối cùng phải chứa TRAIN_SUBDIR.
import zipfile
source = Path(DATA_SOURCE)
if source.is_file():
    assert zipfile.is_zipfile(source), 'DATA_SOURCE phải là ZIP hoặc thư mục'
    import hashlib
    source_key = hashlib.sha256((str(source.resolve()) + str(source.stat().st_size) + str(source.stat().st_mtime_ns)).encode()).hexdigest()[:16]
    extract_dir = Path('/tmp/sshr-data') / (DATASET + '-' + source_key)
    marker = extract_dir / '.complete'
    if not marker.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                target = (extract_dir / member.filename).resolve()
                if not target.is_relative_to(extract_dir.resolve()):
                    raise ValueError('ZIP path không hợp lệ')
            archive.extractall(extract_dir)
        marker.touch()
    candidates = [extract_dir] + [p for p in extract_dir.rglob('*') if p.is_dir() and '__MACOSX' not in p.parts]
    roots = [p for p in candidates if (p / TRAIN_SUBDIR).is_dir()]
    assert len(roots) == 1, f'Tìm được {len(roots)} dataset root; kiểm tra TRAIN_SUBDIR.'
    DATA_ROOT = roots[0]
else:
    DATA_ROOT = source
TRAIN_ROOT = DATA_ROOT / TRAIN_SUBDIR
assert TRAIN_ROOT.is_dir(), TRAIN_ROOT
print('Train root:', TRAIN_ROOT)
print('Images:', sum(p.suffix.lower() in ('.png', '.jpg') for p in TRAIN_ROOT.rglob('*')))


In [ ]:
# Chuyển initialization .params một lần. Khi resume thì bỏ qua hoàn toàn.
INIT = Path(WEIGHTS)
if not RESUME and INIT.suffix == '.params':
    converted = OUT / 'imagenet_init.pth'
    if not converted.exists():
        legacy = Path('/tmp/sshr-mxnet-env')
        run([UV, 'venv', '--python', '3.10', '--allow-existing', legacy])
        legacy_python = legacy / 'bin/python'
        run([UV, 'pip', 'install', '--python', legacy_python, 'numpy==1.23.5', 'mxnet==1.9.1'])
        converter_source = "def convert_mxnet_to_torch(filename):\n    import mxnet\n\n    save_dict = mxnet.nd.load(filename)\n\n    renamed_dict = dict()\n\n    bn_param_mx_pt = {'beta': 'bias', 'gamma': 'weight', 'mean': 'running_mean', 'var': 'running_var'}\n\n    for k, v in save_dict.items():\n\n        v = v.asnumpy()\n        toks = k.split('_')\n\n        if 'conv1a' in toks[0]:\n            renamed_dict['conv1a.weight'] = v\n\n        elif 'linear1000' in toks[0]:\n            pass\n\n        elif 'branch' in toks[1]:\n\n            pt_name = []\n\n            if toks[0][-1] != 'a':\n                pt_name.append('b' + toks[0][-3] + '_' + toks[0][-1])\n            else:\n                pt_name.append('b' + toks[0][-2])\n\n            if 'res' in toks[0]:\n                layer_type = 'conv'\n                last_name = 'weight'\n\n            else:  # 'bn' in toks[0]:\n                layer_type = 'bn'\n                last_name = bn_param_mx_pt[toks[-1]]\n\n            pt_name.append(layer_type + '_' + toks[1])\n\n            pt_name.append(last_name)\n\n            torch_name = '.'.join(pt_name)\n            renamed_dict[torch_name] = v\n\n        else:\n            last_name = bn_param_mx_pt[toks[-1]]\n            renamed_dict['bn7.' + last_name] = v\n\n    return renamed_dict"
        script = REPO / '_convert_numpy.py'
        script.write_text(converter_source + "\nimport numpy as np, sys\nnp.savez(sys.argv[2], **convert_mxnet_to_torch(sys.argv[1]))\n")
        arrays = Path('/tmp/sshr-init.npz')
        run([legacy_python, script, INIT, arrays])
        OUT.mkdir(parents=True, exist_ok=True)
        run([PYTHON, '-c',
             "import numpy as np, torch, sys; a=np.load(sys.argv[1]); torch.save({k:torch.from_numpy(a[k]) for k in a.files},sys.argv[2])",
             arrays, converted])
    INIT = converted
print('Initialization:', INIT if not RESUME else 'Không cần — dùng full resume checkpoint')


In [ ]:
# Kiểm tra parser nhãn thực tế và số batch trước khi dùng GPU lâu.
probe = """
import sys
from tool.GenDataset import Stage1_TrainDataset
from torch.utils.data import DataLoader
x = Stage1_TrainDataset(sys.argv[1], dataset=sys.argv[2], img_size=224)
print('Parsed training images:', len(x), flush=True)
assert len(x) >= int(sys.argv[3]), 'Không đủ ảnh hợp lệ; kiểm tra root và định dạng nhãn filename'
_, image, label = x[0]
print('Image:', image.shape, 'label:', label, flush=True)
assert len(label) == 4
"""
run([PYTHON, '-c', probe, TRAIN_ROOT, DATASET, BATCH_SIZE], cwd=REPO)


In [ ]:
# Train một GPU. Không tự giảm batch khi OOM vì sẽ thay đổi công thức train/resume.
OUT.mkdir(parents=True, exist_ok=True)
local_resume = OUT / 'training_resume.pth'
if not RESUME and local_resume.exists():
    raise RuntimeError('Output đã có checkpoint. Chạy lại cell cấu hình để train mới, hoặc điền RESUME rõ ràng.')
if RESUME and not local_resume.exists():
    shutil.copy2(RESUME, local_resume)
resume_path = str(local_resume) if RESUME else ''
print('Tiếp tục checkpoint' if resume_path else 'TRAIN TỪ ĐẦU (ImageNet initialization)', flush=True)
print('Output:', OUT, flush=True)
config = dict(dataset=DATASET, hf_repo=HF_REPO_ID, hf_revision=HF_RESOLVED_REVISION,
              seed=SEED, max_epochs=MAX_EPOCHS,
              batch_size=BATCH_SIZE, amp=AMP_USED, num_workers=NUM_WORKERS,
              train_root=str(TRAIN_ROOT), source_resume=resume_path,
              session_epochs=SESSION_EPOCHS, session_hours=SESSION_HOURS)
(OUT / 'kaggle_config.json').write_text(json.dumps(config, indent=2))
with (OUT / 'environment.txt').open('w') as f:
    subprocess.run([str(UV), 'pip', 'freeze', '--python', str(PYTHON)], stdout=f, check=True)
cmd = [PYTHON, '-u', 'train_sshr.py',
       '--dataset', DATASET, '--seed', SEED, '--max_epoches', MAX_EPOCHS,
       '--weights', INIT, '--trainroot', TRAIN_ROOT, '--save_folder', OUT,
       '--eval_every', 0, '--save-last-k-checkpoints', 0,
       '--n_class', 4, '--img_size', 224, '--num_workers', NUM_WORKERS,
       '--batch_size', BATCH_SIZE, '--amp-dtype', AMP_USED,
       '--session-epochs', SESSION_EPOCHS, '--session-hours', SESSION_HOURS]
if resume_path:
    cmd += ['--resume', resume_path]
env = os.environ.copy()
env.update(OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', MPLBACKEND='Agg',
           PYTORCH_ALLOC_CONF='expandable_segments:True')
start = time.monotonic()
with (OUT / 'train.log').open('a') as log:
    process = subprocess.Popen(list(map(str, cmd)), cwd=REPO, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
assert status == 0, f'Train thất bại ({status}); xem train.log. OOM: giảm batch rồi bắt đầu run mới.'
print(f'Thời gian train phiên này: {(time.monotonic()-start)/3600:.2f} giờ')
print('Lưu output của version này và gắn làm Input phiên tiếp:', local_resume)


In [ ]:
# Đọc tiến độ bằng interpreter riêng (kernel Kaggle không cần cùng PyTorch).
summary = subprocess.check_output([str(PYTHON), '-c',
    "import torch,sys,json; s=torch.load(sys.argv[1],map_location='cpu',weights_only=False); print(json.dumps({'completed_epochs':s['epoch'],'global_step':s['global_step']}))",
    str(local_resume)], text=True)
progress = json.loads(summary)
print(progress)
if RUN_EVAL and progress['completed_epochs'] >= MAX_EPOCHS:
    assert (DATA_ROOT / TEST_SUBDIR / 'img').is_dir()
    assert (DATA_ROOT / TEST_SUBDIR / 'mask').is_dir()
    run([PYTHON, '-u', 'train_sshr.py', '--evaluate-only', '--dataset', DATASET,
         '--seed', SEED, '--weights', OUT / 'stage1_last.pth',
         '--testroot', DATA_ROOT / TEST_SUBDIR, '--n_class', 4, '--img_size', 224,
         '--num_workers', NUM_WORKERS, '--amp-dtype', AMP_USED], cwd=REPO, env=env)
elif RUN_EVAL:
    print('Chưa đủ tổng epoch; bỏ qua evaluation.')
print('Output files:')
for p in sorted(OUT.iterdir()):
    print(p.name, f'{p.stat().st_size / 2**20:.1f} MiB')


## Chạy phiên tiếp theo

- Save Version và giữ output. Add Input notebook output này vào notebook phiên sau.
- Đặt `RESUME='/kaggle/input/.../checkpoints_luad_seed11_<RUN_ID>/training_resume.pth'` theo đường dẫn thật.
- Giữ nguyên dataset, seed, batch, AMP, số workers và `MAX_EPOCHS=25`. Có thể đổi `SESSION_EPOCHS` / `SESSION_HOURS`.
- `stage1_last.pth` chỉ dùng inference/weights; `training_resume.pth` mới chứa optimizer, scaler, LR step và random state để tiếp tục training.
- `BATCH_SIZE=4` khác batch mặc định 20 của tác giả: đây là lựa chọn tiết kiệm VRAM, chưa được benchmark về chất lượng. Muốn tái hiện cấu hình tác giả, thử batch 20 từ đầu nếu GPU đủ RAM. Không tự scale LR.
- Dataset thứ tự đã được chuẩn hóa khi resume; cuDNN vẫn dùng cấu hình nhanh của repo nên không bảo đảm kết quả bit-for-bit giữa các GPU.
- Nếu phiên bị dừng giữa epoch, resume sẽ chạy lại từ cuối epoch đã lưu gần nhất. Epoch đầu chưa hoàn tất thì chưa có full checkpoint.
- Không xóa/tạo lại môi trường train trong khi process đang chạy. `RUN_EVAL=True` chỉ đánh giá sau tổng epoch; hãy dành đủ quota cho bước này.

- Notebook in commit SHA đã tải và lưu vào `kaggle_config.json`. Khi resume, đặt `HF_REVISION` bằng SHA đó để dùng đúng phiên bản dataset.
